In [11]:
import numpy as np
import pandas as pd
import sys
sys.path.append("C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code/")
from bmresearch_tools import BMR_load, BMR_plot, get_metadata
%matplotlib widget
import matplotlib.pyplot as plt
import datetime
import h5py
import os

In [3]:
# read in EKG 
bm_filepath = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/old/BMR_002_v1.h5'
output_h5_path = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/ECG_002.h5'

In [3]:
bmr_studyid_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/'
ecg_studyid_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/'

bm_files = os.listdir(bmr_studyid_dir)
for bm_file in bm_files:
    bm_filepath = os.path.join(bmr_studyid_dir, bm_file)
    output_h5_path = os.path.join(ecg_studyid_dir, bm_file.replace('BMR','ECG'))
    # do the resampling for this subject:
    resample_ecg(bm_filepath, output_h5_path)

NameError: name 'resample_ecg' is not defined

In [370]:
'//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG/ECG_002.h5'.replace('BMR','ECG')

'//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/ECG_resampled_ECG/ECG_002.h5'

In [4]:
def resample_ecg(bm_filepath, output_h5_path):
    
    signals_in_bm_file = get_metadata(bm_filepath)
    ekg_leads = ['I','II','III', 'IIII','V','VI','VII','VIII']
    ekg_leads = [x for x in ekg_leads if x in signals_in_bm_file]
    
    # resample and save bedmaster ecg data for each lead:
    
    for lead in ekg_leads:
        # load the data:
        lead_data = BMR_load(bm_filepath, signals = lead, loadEvents=1)[lead]

        # resample to 240Hz:
        lead_data.set_index('datetime', inplace=True)
        lead_data = lead_data[['signal']]
        lead_data_original_start = lead_data.index[0]
        lead_data = lead_data.resample(datetime.timedelta(seconds = 1/240)).mean() 
        # lead_data_resampled_start = lead_data.index[0]
        # difference_start = lead_data_original_start - lead_data_resampled_start
        lead_data = lead_data.interpolate(method='pchip', order =3)   #  limit = 4. limit can be used to not interpolate gaps. however, matlab cardiovascular toolbox does expect continuous data. therefore interpolate also gaps.
        # lead_data.index = lead_data.index + difference_start
        datetime_array = np.array([lead_data_original_start.year, lead_data_original_start.month, lead_data_original_start.day, lead_data_original_start.hour, lead_data_original_start.minute, lead_data_original_start.second, lead_data_original_start.microsecond]) 

        # and save:
        chunk_size = 64 
        with h5py.File(output_h5_path, 'a') as f:
            dset_ekg = f.create_dataset(lead, shape=(lead_data.shape[0],), maxshape=(None,),
                                      chunks=(chunk_size,), dtype='float64')
            dset_ekg[:] = lead_data.signal.astype('float64')

            dset_startdatetime = f.create_dataset('startdatetime', shape=datetime_array.shape, maxshape=(None,),
                                      chunks=(chunk_size,), dtype='int64')
            dset_startdatetime[:] = datetime_array.astype('int64')


In [329]:
resample_ecg(bm_filepath, output_h5_path)

In [4]:
signals_in_bm_file = get_metadata(bm_filepath)
print(signals_in_bm_file)
ekg_leads = ['I','II','III', 'IIII','V','VI','VII','VIII']
ekg_leads = [x for x in ekg_leads if x in signals_in_bm_file]
print(ekg_leads)

['ART1', 'ART1D', 'ART1M', 'ART1R', 'ART1S', 'CUFF', 'CVP2', 'FLWR', 'FLWTRIG', 'HR', 'I', 'I:E', 'II', 'III', 'IN_HLD', 'MAWP', 'MV', 'NBPD', 'NBPM', 'NBPS', 'PA2 ', 'PEEP', 'PIP', 'PPLAT', 'PRSSUP', 'PTRR', 'PVC', 'SENS', 'SETFIO2', 'SETIE', 'SETPCP', 'SETTV', 'SPO2', 'SPO2%', 'SPO2R', 'SPONTMV', 'STAVF', 'STAVL', 'STAVR', 'STI', 'STII', 'STIII', 'STV', 'STV1', 'STV2', 'STV3', 'STV4', 'STV5', 'STV6', 'TV', 'V', 'Vent Rate']
['I', 'II', 'III', 'V']


In [19]:
signals_in_bm_file = get_metadata(bm_filepath)
print(signals_in_bm_file)
wv_signals = ['I','II','III', 'IIII','V','VI','VII','VIII','SPO2','CO2','ART1','ART2','RESP']
wv_signals = [x for x in wv_signals if x in signals_in_bm_file]
print(wv_signals)

['APNEA', 'ART1', 'ART1D', 'ART1M', 'ART1R', 'ART1S', 'ART2D', 'ART2M', 'ART2R', 'ART2S', 'CUFF', 'CVP2', 'FLWR', 'FLWTRIG', 'HR', 'I', 'I:E', 'II', 'III', 'IN_HLD', 'MAWP', 'MV', 'NBPD', 'NBPM', 'NBPS', 'PEEP', 'PIP', 'PPLAT', 'PRSSUP', 'PTRR', 'PVC', 'RESP', 'RR', 'SENS', 'SETFIO2', 'SETIE', 'SETPCP', 'SETTV', 'SPO2', 'SPO2%', 'SPO2R', 'SPONTMV', 'STAVF', 'STAVL', 'STAVR', 'STI', 'STII', 'STIII', 'STV', 'TV', 'V', 'Vent Rate']
['I', 'II', 'III', 'V', 'SPO2', 'ART1', 'RESP']


In [5]:
# resample and save bedmaster ecg data for each lead:

lead = ekg_leads[0]
# load the data:
lead_data_orig = BMR_load(bm_filepath, signals = lead, loadEvents=1)[lead]

lead_data = lead_data_orig.copy()


In [6]:
# lead_data1 = lead_data.iloc[:1000]
lead_data1 = pd.concat([lead_data.iloc[:1000], lead_data.iloc[10000:11000]])
# resample to 240Hz:
lead_data1.set_index('datetime', inplace=True)
lead_data1 = lead_data1[['signal']]

lead_data_original_start = lead_data1.index[0]


In [22]:
plt.close()
plt.plot(lead_data1.index, lead_data1.signal)
plt.scatter(lead_data1.index, lead_data1.signal, s=2,c='r')
plt.xlim([min(lead_data1.index), max(lead_data1.index)])
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [24]:
lead_data = lead_data1.resample(datetime.timedelta(seconds = 1/240)).mean() 
plt.close()
plt.plot(lead_data.index, lead_data.signal)
plt.scatter(lead_data.index, lead_data.signal, s=2,c='r')
plt.xlim([min(lead_data.index), max(lead_data.index)])
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [32]:
lead_data1.shape

(2000, 1)

In [33]:
lead_data.shape

(10999, 1)

In [31]:
print(lead_data.iloc[2000:])

                            signal
datetime                          
2018-06-11 14:27:08.332767     NaN
2018-06-11 14:27:08.336934     NaN
2018-06-11 14:27:08.341101     NaN
2018-06-11 14:27:08.345268     NaN
2018-06-11 14:27:08.349435     NaN
...                            ...
2018-06-11 14:27:45.810765   -15.0
2018-06-11 14:27:45.814932    -8.0
2018-06-11 14:27:45.819099   -10.0
2018-06-11 14:27:45.823266    -5.0
2018-06-11 14:27:45.827433   -12.0

[8999 rows x 1 columns]


In [41]:
lead_data_interp_limit = lead_data.interpolate(method='pchip', order =3, limit = 4)
lead_data_interp = lead_data.interpolate(method='pchip', order =3)

In [42]:
plt.close()
plt.plot(lead_data_interp.index, lead_data_interp.signal)
plt.plot(lead_data_interp_limit.index, lead_data_interp_limit.signal)
# plt.scatter(lead_data_interp_limit.index, lead_data_interp_limit.signal, s=2,c='r')
plt.xlim([min(lead_data_interp.index), max(lead_data_interp_limit.index)])
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [39]:
lead_data_interp.shape

(10999, 1)

In [40]:
print(lead_data_interp.iloc[2000:])

                            signal
datetime                          
2018-06-11 14:27:08.332767     NaN
2018-06-11 14:27:08.336934     NaN
2018-06-11 14:27:08.341101     NaN
2018-06-11 14:27:08.345268     NaN
2018-06-11 14:27:08.349435     NaN
...                            ...
2018-06-11 14:27:45.810765   -15.0
2018-06-11 14:27:45.814932    -8.0
2018-06-11 14:27:45.819099   -10.0
2018-06-11 14:27:45.823266    -5.0
2018-06-11 14:27:45.827433   -12.0

[8999 rows x 1 columns]


In [ ]:
lead_data = lead_data1.resample(datetime.timedelta(seconds = 1/240)).mean() 
# lead_data_resampled_start = lead_data.index[0]
# difference_start = lead_data_original_start - lead_data_resampled_start
lead_data = lead_data.interpolate(method='pchip', order =3)
# lead_data.index = lead_data.index + difference_start
datetime_array = np.array([lead_data_original_start.year, lead_data_original_start.month, lead_data_original_start.day, lead_data_original_start.hour, lead_data_original_start.minute, lead_data_original_start.second, lead_data_original_start.microsecond]) 
plt.plot(lead_data1.index, lead_data1.signal)
plt.plot(lead_data.index, lead_data.signal)
plt.show()

In [330]:
# EKG data resampled to .h5:
lead_data.shape

(9999, 1)

In [352]:
lead_data.head()

,signal
datetime,
2018-06-11 14:26:59.998767,-11.0
2018-06-11 14:27:00.002934,7.0
2018-06-11 14:27:00.007101,-5.0
2018-06-11 14:27:00.011268,-32.0
2018-06-11 14:27:00.015435,-9.0


In [42]:
lead_data.head()

,signal,posix,event
datetime,,,
2018-06-11 14:27:00.000000,-11.0,1.528742e+09,0
2018-06-11 14:27:00.004167,7.0,1.528742e+09,0
2018-06-11 14:27:00.008333,-5.0,1.528742e+09,0
2018-06-11 14:27:00.012500,-32.0,1.528742e+09,0
2018-06-11 14:27:00.016667,-9.0,1.528742e+09,0


In [44]:
lead_data.shape

(27250756, 3)

In [45]:
lead_data_full = lead_data.copy()

In [46]:
lead_data = lead_data.iloc[:10000,:]

In [82]:
fs_blocks = [1/np.mean(np.diff(lead_data_full.posix[x:x+10000])) for x in range(lead_data_full.shape[0] - 10000) ]

KeyboardInterrupt: 

In [83]:
np.where(np.array(fs_blocks) < 150)[0]

array([], dtype=int64)

In [68]:
1/np.mean(np.diff(lead_data.posix[:100]))

240.00005548651853

In [300]:
seconds = 1/480
seconds

0.0020833333333333333

In [301]:
nanoseconds = seconds*(10*9)
nanoseconds

0.1875

In [302]:
lead_data = lead_data[['signal']]

In [303]:
lead_data.head()

,signal
datetime,
2018-06-11 14:27:00.000000,-11.0
2018-06-11 14:27:00.004167,7.0
2018-06-11 14:27:00.008333,-5.0
2018-06-11 14:27:00.012500,-32.0
2018-06-11 14:27:00.016667,-9.0


In [310]:
lead_data.shape

(10000, 1)

In [312]:
lead_data_gap = pd.concat([lead_data.iloc[:300,:], lead_data.iloc[700:]])
lead_data_gap

,signal
datetime,
2018-06-11 14:27:00.000000,-11.0
2018-06-11 14:27:00.004167,7.0
2018-06-11 14:27:00.008333,-5.0
2018-06-11 14:27:00.012500,-32.0
2018-06-11 14:27:00.016667,-9.0
...,...
2018-06-11 14:27:41.645833,-55.0
2018-06-11 14:27:41.650000,-29.0
2018-06-11 14:27:41.654167,-37.0


In [313]:
lead_data_original_start = lead_data.index[0]
lead_data_240 = lead_data.resample(datetime.timedelta(seconds = 1/240)).mean() # .interpolate(method='spline', order =2, limit_direction='forward', axis=0)
lead_data_240_start = lead_data_240.index[0]
lead_data_gap_240 = lead_data_gap.resample(datetime.timedelta(seconds = 1/240)).mean() # .interpolate(method='spline', order =2, limit_direction='forward', axis=0)

In [305]:
difference_start = lead_data_original_start - lead_data_240_start

In [306]:
lead_data_240 = lead_data_240.interpolate(method='pchip', order =3, limit = 4)

In [307]:
lead_data_240.index = lead_data_240.index + difference_start

In [308]:
lead_data_240.head()

,signal
datetime,
2018-06-11 14:27:00.000000,-11.0
2018-06-11 14:27:00.004167,7.0
2018-06-11 14:27:00.008334,-5.0
2018-06-11 14:27:00.012501,-32.0
2018-06-11 14:27:00.016668,-9.0


In [315]:
plt.close()
plt.plot(lead_data.index, lead_data.signal)
plt.plot(lead_data_240.index, lead_data_240.signal)
plt.plot(lead_data_gap_240.index, lead_data_gap_240.signal)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

,signal
datetime,
2018-06-11 14:26:59.998767,-11.0
2018-06-11 14:27:00.002934,7.0
2018-06-11 14:27:00.007101,-5.0
2018-06-11 14:27:00.011268,-32.0
2018-06-11 14:27:00.015435,-9.0
...,...
2018-06-11 14:27:41.643765,-55.0
2018-06-11 14:27:41.647932,-29.0
2018-06-11 14:27:41.652099,-37.0
